In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

import ee
import geemap
import geopandas as gpd
from utils.variables import PROJECT, ANALYSIS_END_YR, MATCHED_GRIDS_TEST

from absolute_effectiveness.site_selector import SiteSelector
from absolute_effectiveness.data_processor import DataProcessor
from relative_effectiveness.metrics_per_cell import (
    RelativeHabitatConditionAnalyzer,
    RelativeHabitatLossAnalyzer,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()
processor = DataProcessor.from_gee_defaults()
condition_analyzer = RelativeHabitatConditionAnalyzer()
loss_analyzer = RelativeHabitatLossAnalyzer()

In [ ]:
# Load matched_grids (output of ps_model.ipynb) and derive site-specific context
site_id = 1543

matched_grids_gdf = gpd.read_parquet(f"../{MATCHED_GRIDS_TEST}").to_crs(epsg=4326)
matched_grids = geemap.geopandas_to_ee(matched_grids_gdf)

test_sites = site_selector.get_test_sites()
START_YR = site_selector.set_start_yr(test_sites, site_id)
site_geom = site_selector.get_site_geom(test_sites, site_id)

site_selector.check_start_yr(START_YR)

In [ ]:
# Process all input datasets, scoped to the bounds of the matched cells
# (covers PA + control buffer, since control cells live outside the PA)
GLC_processed = processor.process_glc(matched_grids, START_YR)
GPW_processed = processor.process_gpw(START_YR)
NFW_processed = processor.process_nfw(matched_grids)
HGFC_processed = processor.process_hgfc(START_YR)

In [ ]:
# Build habitat / intactness rasters and score Habitat Extent, Intactness, and Condition per cell
habitat_raster = condition_analyzer.get_habitat_raster(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed
)
exp_kernel = condition_analyzer.build_kernel()
intactness_raster = condition_analyzer.get_intactness_raster(
    habitat_raster, matched_grids.geometry(), exp_kernel
)

scored = condition_analyzer.calc_extent_score_per_cell(habitat_raster, matched_grids)
scored = condition_analyzer.calc_intactness_score_per_cell(intactness_raster, scored)
scored = condition_analyzer.calc_condition_score_per_cell(scored)

In [ ]:
# Build habitat-loss rasters and score Habitat Loss per cell
habitat_loss_raster, habitat_start_raster = loss_analyzer.get_habitat_loss_raster(
    GLC_processed, GPW_processed, HGFC_processed, START_YR
)
scored = loss_analyzer.calc_loss_score_per_cell(
    habitat_loss_raster, habitat_start_raster, scored
)

print(f"Site ID: {site_id}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"Scored cells: {scored.size().getInfo()}")
print("\nFirst feature properties:")
print(scored.first().getInfo()["properties"])

In [ ]:
# ============================================================================
# VISUALIZATION
# ============================================================================

from utils.variables import GLC_PALETTE

score_palette = ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]
score_viz = {"min": 0, "max": 1, "palette": score_palette}

def score_image(fc, prop):
    """Rasterize a numeric per-feature score so it can be displayed with a palette."""
    return fc.reduceToImage(properties=[prop], reducer=ee.Reducer.first())

Map = geemap.Map()
Map.add_basemap("CartoDB.DarkMatter")
Map.add_basemap("Esri.WorldImagery")

# Map.addLayer(GLC_processed.select("GLC_2022"), {"min": 0, "max": 36, "palette": GLC_PALETTE}, "Global Land Cover", 0)
# Map.addLayer(GPW_processed.select("GPW_2022").selfMask(), {"min": 1, "max": 2, "palette": ["#ffcd73", "#ff9916"]}, "Global Pasture Watch", 0)
# Map.addLayer(NFW_processed.selfMask(), {"palette": "teal"}, "Natural Forests of the World", 0)
# Map.addLayer(HGFC_processed.selfMask(), {"min": START_YR - 2000, "max": ANALYSIS_END_YR - 2000, "palette": ["yellow", "red"]}, "Hansen Global Forest Change", 0)

Map.addLayer(habitat_raster, {"palette": GLC_PALETTE}, "Habitat extent")
Map.addLayer(intactness_raster, {"min": 0, "max": 1, "palette": ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]}, "Habitat intactness")

Map.addLayer(site_geom, {"color": "yellow"}, "PA boundary", False)
Map.addLayer(matched_grids, {"color": "white"}, "Matched cells (outline)", False)

Map.addLayer(score_image(scored, "extent_score"), score_viz, "Extent Score")
Map.addLayer(score_image(scored, "intactness_score"), score_viz, "Intactness Score")
Map.addLayer(score_image(scored, "condition_score"), score_viz, "Condition Score")
Map.addLayer(score_image(scored, "loss_score"), score_viz, "Loss Score")

Map.centerObject(matched_grids)

Map